# F02-P2 Threat

**Ecosystem Threat: Dryland Forest, Mangrove, Peatland**

Showcasing the disturbance in three different ecosystems, mainly on forest disturbances but not limited to hydrological disturbances for peatland ecosystems. F02-P3 Threat answers “where is the most disturbed area and its drivers”.

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: All ecosystems section.

**Known limit of that scope.** Forest disturbance is an internal agreement term for forest degradation with the structural change across 10 years. However, the 10 years of forest degradation is too long, the next v3.1 this will be revised into annual or maximum 5 years analysis.

## Setup

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio

from pyproj import Geod
from rasterio.mask import raster_geometry_mask

---
## 3.1 All Ecosystem (Overview)

Reports the total ecosystem and disturbed area across three different ecosystem in hectare.

**Data.** `forest_disturbance_v3.tif: any pixel > 0 = disturbed` following C. Bourgoin, 2024. The 
methodology published following JRC-TMF data on degraded and undisturbed forest. Scene approach
focusses on structural decline on forest using TCC and TCH by SIGnal forest cover. However, the drivers
of degradation only limited to selective logging and forest fire

**Calibration warning.** The 0 to 3 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published JRC-TMF. 
**Decisions locked.**

Structural decline within retained forest, assessed as canopy height deficit
relative to an undisturbed reference population defined at >=120 m from any
disturbed forest. This addresses the growing stock and biomass marker of FAO
(2011, FRA Working Paper 177). It is not a complete assessment of forest
degradation as defined by FAO, and no globally agreed operational definition
currently exists. Detection threshold set at the 5th percentile of reference
population height change, giving a nominal 5% false-positive rate. Results
are reported as area statistics by stratum; per-pixel interpretation is not
supported at 30 m given a product RMSE of 6.6-9.1 m.

**Example render**

## Ecosystem Summary

| Summary | Value |
|---|---:|
| **Total ecosystem area** | **78,450.00 ha** |
| **Total disturbed area** | **18,760.00 ha (23.91%)** |

### Ecosystem Breakdown

| Ecosystem | Area (ha) | % Total | Disturbed (ha) | Disturbed % |
|---|---:|---:|---:|---:|
| **Dryland forest** | 31,380.00 | 40.00% | 7,845.00 | 25.00% |
| **Mangrove** | 27,457.50 | 35.00% | 6,589.80 | 24.00% |
| **Peatland** | 19,612.50 | 25.00% | 7,845.00 | 40.00% |

In [ ]:
def pixel_area_by_row(transform, height):
    """
    Calculate pixel area in hectares by raster row.
    Suitable for EPSG:4326.
    """

    pixel_width = abs(transform.a)

    row_areas = np.zeros(height)

    for row in range(height):

        north = transform.f + row * transform.e
        south = north + transform.e

        west = transform.c
        east = west + pixel_width

        area_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        row_areas[row] = abs(area_m2) / 10000

    return row_areas


def calculate_mask_area_ha(binary_mask, transform):
    """
    Calculate area of True pixels in hectares.
    """

    row_areas = pixel_area_by_row(
        transform,
        binary_mask.shape[0]
    )

    pixels_per_row = binary_mask.sum(axis=1)

    return float(
        np.sum(
            pixels_per_row * row_areas
        )
    )


def read_masked_raster(
    raster_path,
    aoi_geometry,
    aoi_crs
):
    """
    Read only the AOI window from a raster.
    """

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        if src.crs != aoi_crs:

            raster_aoi = (
                gpd.GeoSeries(
                    [aoi_geometry],
                    crs=aoi_crs
                )
                .to_crs(src.crs)
                .iloc[0]
            )

        else:

            raster_aoi = aoi_geometry


        outside_mask, transform, window = (
            raster_geometry_mask(
                src,
                [raster_aoi.__geo_interface__],
                crop=True,
                all_touched=False
            )
        )


        data = src.read(
            1,
            window=window,
            masked=True
        )


        valid_mask = (
            ~outside_mask
            & ~np.ma.getmaskarray(data)
        )


        return (
            data.data,
            valid_mask,
            transform,
            src.crs
        )


# =============================================================================
# Main Analysis
# =============================================================================

def analyze_all_ecosystem():

    # -------------------------------------------------------------------------
    # 1. Read AOI
    # -------------------------------------------------------------------------

    aoi = gpd.read_file(
        USER_AOI
    )

    if aoi.empty:
        raise ValueError(
            "AOI contains no features."
        )

    if aoi.crs is None:
        raise ValueError(
            "AOI has no CRS."
        )


    valid_geometry = aoi.geometry[
        aoi.geometry.notna()
        & ~aoi.geometry.is_empty
    ]


    if valid_geometry.empty:
        raise ValueError(
            "AOI contains no valid geometry."
        )


    aoi_geometry = (
        valid_geometry.union_all()
    )


    # -------------------------------------------------------------------------
    # 2. Read Ecosystem Raster
    # -------------------------------------------------------------------------

    (
        ecosystem_data,
        ecosystem_valid,
        ecosystem_transform,
        ecosystem_crs
    ) = read_masked_raster(
        ECOSYSTEM_RASTER,
        aoi_geometry,
        aoi.crs
    )


    # -------------------------------------------------------------------------
    # 3. Read Disturbance Raster
    # -------------------------------------------------------------------------

    (
        disturbance_data,
        disturbance_valid,
        disturbance_transform,
        disturbance_crs
    ) = read_masked_raster(
        DISTURBANCE_RASTER,
        aoi_geometry,
        aoi.crs
    )


    # -------------------------------------------------------------------------
    # IMPORTANT:
    # Both rasters must share the same clipped grid for direct boolean masking.
    # -------------------------------------------------------------------------

    if ecosystem_data.shape != disturbance_data.shape:

        raise ValueError(
            "Ecosystem and disturbance raster grids do not match "
            "after clipping."
        )


    if not np.allclose(
        ecosystem_transform,
        disturbance_transform
    ):

        raise ValueError(
            "Ecosystem and disturbance raster transforms do not match."
        )


    # -------------------------------------------------------------------------
    # 4. Total Ecosystem Mask
    # -------------------------------------------------------------------------

    total_ecosystem_mask = (
        ecosystem_valid
        & np.isin(
            ecosystem_data,
            list(
                ECOSYSTEM_CLASSES.keys()
            )
        )
    )


    total_ecosystem_area_ha = (
        calculate_mask_area_ha(
            total_ecosystem_mask,
            ecosystem_transform
        )
    )


    # -------------------------------------------------------------------------
    # 5. Total Disturbed Area
    # -------------------------------------------------------------------------

    disturbance_mask = (
        disturbance_valid
        & (disturbance_data > 0)
    )


    total_disturbed_mask = (
        total_ecosystem_mask
        & disturbance_mask
    )


    total_disturbed_area_ha = (
        calculate_mask_area_ha(
            total_disturbed_mask,
            ecosystem_transform
        )
    )


    total_disturbed_percentage = (
        total_disturbed_area_ha
        / total_ecosystem_area_ha
        * 100
        if total_ecosystem_area_ha > 0
        else 0
    )


    # -------------------------------------------------------------------------
    # 6. Ecosystem Breakdown
    # -------------------------------------------------------------------------

    ecosystem_results = {}


    for class_value, class_name in (
        ECOSYSTEM_CLASSES.items()
    ):

        ecosystem_mask = (
            ecosystem_valid
            & (ecosystem_data == class_value)
        )


        ecosystem_area_ha = (
            calculate_mask_area_ha(
                ecosystem_mask,
                ecosystem_transform
            )
        )


        ecosystem_percentage = (
            ecosystem_area_ha
            / total_ecosystem_area_ha
            * 100
            if total_ecosystem_area_ha > 0
            else 0
        )


        # ---------------------------------------------------------------------
        # Disturbed within ecosystem
        # ---------------------------------------------------------------------

        ecosystem_disturbed_mask = (
            ecosystem_mask
            & disturbance_mask
        )


        ecosystem_disturbed_area_ha = (
            calculate_mask_area_ha(
                ecosystem_disturbed_mask,
                ecosystem_transform
            )
        )


        ecosystem_disturbed_percentage = (
            ecosystem_disturbed_area_ha
            / ecosystem_area_ha
            * 100
            if ecosystem_area_ha > 0
            else 0
        )


        ecosystem_results[
            class_name
        ] = {

            "area_ha":
                round(
                    ecosystem_area_ha,
                    2
                ),

            "percentage_total":
                round(
                    ecosystem_percentage,
                    2
                ),

            "disturbed_area_ha":
                round(
                    ecosystem_disturbed_area_ha,
                    2
                ),

            "disturbed_percentage":
                round(
                    ecosystem_disturbed_percentage,
                    2
                )
        }


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "total_ecosystem_area_ha":
            round(
                total_ecosystem_area_ha,
                2
            ),

        "total_disturbed_area_ha":
            round(
                total_disturbed_area_ha,
                2
            ),

        "total_disturbed_percentage":
            round(
                total_disturbed_percentage,
                2
            ),

        "ecosystems":
            ecosystem_results
    }


# =============================================================================
# Run
# =============================================================================

result = analyze_all_ecosystem()


# =============================================================================
# Display
# =============================================================================

print()
print("=" * 70)
print("ALL ECOSYSTEM SCREENING")
print("=" * 70)

print(
    f"Total ecosystem area : "
    f"{result['total_ecosystem_area_ha']:,.2f} ha"
)

print(
    f"Total disturbed area : "
    f"{result['total_disturbed_area_ha']:,.2f} ha "
    f"({result['total_disturbed_percentage']:.2f}%)"
)


print()
print("Ecosystem breakdown:")
print("-" * 90)

print(
    f"{'Ecosystem':<20}"
    f"{'Area (ha)':>15}"
    f"{'% Total':>12}"
    f"{'Disturbed (ha)':>20}"
    f"{'Disturbed %':>15}"
)

print("-" * 90)


for ecosystem_name, values in (
    result["ecosystems"].items()
):

    print(
        f"{ecosystem_name:<20}"
        f"{values['area_ha']:>15,.2f}"
        f"{values['percentage_total']:>11.2f}%"
        f"{values['disturbed_area_ha']:>20,.2f}"
        f"{values['disturbed_percentage']:>14.2f}%"
    )

---
## 3.2 Dryland Forest Ecosystem

Reports the drivers of dryland forest disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.** Post-processing steps identified where fire coincided with the clearing of agricultural land, based on the presence of VIIRS fire alerts (within a 500 m buffer around the alert) and a low post-disturbance normalized-burn ration in the following month’s Sentinen-2 composite.
Post-processing steps masking out the area outside forest_disturbance_v3.tif and re-calibrate
with disaster risk from ADPC. However, this data only consider high and very high disaster risk
as part of forest disturbance.

In [ ]:
# =============================================================================
# Forest driver classes
# =============================================================================

FOREST_DRIVERS = {
    1: {
        "name": "Small-scale agriculture",
        "group": "non_natural"
    },
    2: {
        "name": "Small-scale agriculture (fire)",
        "group": "non_natural"
    },
    3: {
        "name": "Large-scale agriculture",
        "group": "non_natural"
    },
    4: {
        "name": "Large-scale agriculture (fire)",
        "group": "non_natural"
    },
    5: {
        "name": "Road development",
        "group": "non_natural"
    },
    6: {
        "name": "Selective logging",
        "group": "non_natural"
    },
    7: {
        "name": "Mining",
        "group": "non_natural"
    },
    8: {
        "name": "Non-productive conversion",
        "group": "other"
    }
}


# =============================================================================
# Natural driver rasters
# =============================================================================

NATURAL_DRIVERS = {
    "Flooding": FLOOD_RISK_RASTER,
    "Forest fire": FIRE_RISK_RASTER,
    "Drought": DROUGHT_RISK_RASTER,
    "Typhoon": TYPHOON_RISK_RASTER,
    "Landslide": LANDSLIDE_RISK_RASTER
}


GEOD = Geod(ellps="WGS84")


# =============================================================================
# Helpers
# =============================================================================

def pixel_area_by_row(transform, height):
    """Calculate EPSG:4326 pixel area by row in hectares."""

    pixel_width = abs(transform.a)
    row_areas = np.zeros(height)

    for row in range(height):

        north = transform.f + row * transform.e
        south = north + transform.e

        west = transform.c
        east = west + pixel_width

        area_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        row_areas[row] = abs(area_m2) / 10000

    return row_areas


def calculate_mask_area_ha(binary_mask, transform):
    """Calculate area of True pixels in hectares."""

    row_areas = pixel_area_by_row(
        transform,
        binary_mask.shape[0]
    )

    pixels_per_row = binary_mask.sum(axis=1)

    return float(
        np.sum(
            pixels_per_row * row_areas
        )
    )


def read_aoi_raster(
    raster_path,
    aoi_geometry,
    aoi_crs
):
    """
    Read only the raster window intersecting the AOI.
    """

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise ValueError(
                f"Raster has no CRS: {raster_path}"
            )

        if src.crs != aoi_crs:

            raster_aoi = (
                gpd.GeoSeries(
                    [aoi_geometry],
                    crs=aoi_crs
                )
                .to_crs(src.crs)
                .iloc[0]
            )

        else:

            raster_aoi = aoi_geometry


        outside_mask, transform, window = (
            raster_geometry_mask(
                src,
                [raster_aoi.__geo_interface__],
                crop=True,
                all_touched=False
            )
        )


        data = src.read(
            1,
            window=window,
            masked=True
        )


        valid_mask = (
            ~outside_mask
            & ~np.ma.getmaskarray(data)
        )


        return (
            data.data,
            valid_mask,
            transform,
            src.crs
        )


def check_same_grid(
    reference_data,
    reference_transform,
    data,
    transform,
    raster_name
):
    """
    Ensure rasters can be directly combined as boolean masks.
    """

    if reference_data.shape != data.shape:

        raise ValueError(
            f"Grid size mismatch: {raster_name}"
        )


    if not np.allclose(
        reference_transform,
        transform
    ):

        raise ValueError(
            f"Grid alignment mismatch: {raster_name}"
        )


# =============================================================================
# Main Analysis
# =============================================================================

def analyze_dryland_forest():

    # -------------------------------------------------------------------------
    # 1. Read AOI
    # -------------------------------------------------------------------------

    aoi = gpd.read_file(
        USER_AOI
    )

    if aoi.empty:
        raise ValueError(
            "AOI contains no features."
        )

    if aoi.crs is None:
        raise ValueError(
            "AOI has no CRS."
        )


    valid_geometry = aoi.geometry[
        aoi.geometry.notna()
        & ~aoi.geometry.is_empty
    ]


    if valid_geometry.empty:
        raise ValueError(
            "AOI contains no valid geometry."
        )


    aoi_geometry = (
        valid_geometry.union_all()
    )


    # -------------------------------------------------------------------------
    # 2. Read ecosystem raster
    # -------------------------------------------------------------------------

    (
        ecosystem_data,
        ecosystem_valid,
        reference_transform,
        reference_crs
    ) = read_aoi_raster(
        ECOSYSTEM_RASTER,
        aoi_geometry,
        aoi.crs
    )


    dryland_mask = (
        ecosystem_valid
        & (ecosystem_data == DRYLAND_CLASS)
    )


    dryland_total_area_ha = (
        calculate_mask_area_ha(
            dryland_mask,
            reference_transform
        )
    )


    # -------------------------------------------------------------------------
    # 3. Historical deforestation
    # -------------------------------------------------------------------------

    (
        historical_data,
        historical_valid,
        historical_transform,
        _
    ) = read_aoi_raster(
        HISTORICAL_DEFORESTATION_RASTER,
        aoi_geometry,
        aoi.crs
    )


    check_same_grid(
        ecosystem_data,
        reference_transform,
        historical_data,
        historical_transform,
        "historical_deforestation_v3.tif"
    )


    remaining_mask = (
        dryland_mask
        & historical_valid
        & (
            historical_data
            == REMAINING_FOREST_CLASS
        )
    )


    forest_loss_mask = (
        dryland_mask
        & historical_valid
        & (
            historical_data
            == FOREST_LOSS_CLASS
        )
    )


    remaining_area_ha = (
        calculate_mask_area_ha(
            remaining_mask,
            reference_transform
        )
    )


    forest_loss_area_ha = (
        calculate_mask_area_ha(
            forest_loss_mask,
            reference_transform
        )
    )


    # -------------------------------------------------------------------------
    # 4. Current forest 2024
    # -------------------------------------------------------------------------

    (
        forest_2024_data,
        forest_2024_valid,
        forest_2024_transform,
        _
    ) = read_aoi_raster(
        FOREST_2024_RASTER,
        aoi_geometry,
        aoi.crs
    )


    check_same_grid(
        ecosystem_data,
        reference_transform,
        forest_2024_data,
        forest_2024_transform,
        "forest_2024_v3.tif"
    )


    current_forest_mask = (
        dryland_mask
        & forest_2024_valid
        & (
            forest_2024_data
            == FOREST_2024_CLASS
        )
    )


    # -------------------------------------------------------------------------
    # 5. Forest disturbance
    # -------------------------------------------------------------------------

    (
        disturbance_data,
        disturbance_valid,
        disturbance_transform,
        _
    ) = read_aoi_raster(
        FOREST_DISTURBANCE_RASTER,
        aoi_geometry,
        aoi.crs
    )


    check_same_grid(
        ecosystem_data,
        reference_transform,
        disturbance_data,
        disturbance_transform,
        "forest_disturbance_v3.tif"
    )


    disturbed_dryland_mask = (
        current_forest_mask
        & disturbance_valid
        & (
            disturbance_data
            > DISTURBANCE_THRESHOLD
        )
    )


    disturbed_area_ha = (
        calculate_mask_area_ha(
            disturbed_dryland_mask,
            reference_transform
        )
    )


    # -------------------------------------------------------------------------
    # 6. Forest gain
    # -------------------------------------------------------------------------

    (
        gain_data,
        gain_valid,
        gain_transform,
        _
    ) = read_aoi_raster(
        FOREST_GAIN_RASTER,
        aoi_geometry,
        aoi.crs
    )


    check_same_grid(
        ecosystem_data,
        reference_transform,
        gain_data,
        gain_transform,
        "forest_gain_v3.tif"
    )


    forest_gain_mask = (
        dryland_mask
        & gain_valid
        & (
            gain_data
            == FOREST_GAIN_CLASS
        )
    )


    forest_gain_area_ha = (
        calculate_mask_area_ha(
            forest_gain_mask,
            reference_transform
        )
    )


    # -------------------------------------------------------------------------
    # 7. Non-natural + other forest drivers
    # -------------------------------------------------------------------------

    (
        driver_data,
        driver_valid,
        driver_transform,
        _
    ) = read_aoi_raster(
        FOREST_DRIVERS_RASTER,
        aoi_geometry,
        aoi.crs
    )


    check_same_grid(
        ecosystem_data,
        reference_transform,
        driver_data,
        driver_transform,
        "forest_drivers_v3.tif"
    )


    non_natural_drivers = []
    other_drivers = []


    known_driver_mask = np.zeros(
        disturbed_dryland_mask.shape,
        dtype=bool
    )


    for driver_value, driver_info in (
        FOREST_DRIVERS.items()
    ):

        driver_presence_mask = (
            disturbed_dryland_mask
            & driver_valid
            & (
                driver_data
                == driver_value
            )
        )


        if np.any(
            driver_presence_mask
        ):

            known_driver_mask |= (
                driver_presence_mask
            )


            if (
                driver_info["group"]
                == "non_natural"
            ):

                non_natural_drivers.append(
                    driver_info["name"]
                )


            elif (
                driver_info["group"]
                == "other"
            ):

                other_drivers.append(
                    driver_info["name"]
                )


    # -------------------------------------------------------------------------
    # 8. Natural risk drivers
    # -------------------------------------------------------------------------

    natural_drivers = []

    natural_driver_mask = np.zeros(
        disturbed_dryland_mask.shape,
        dtype=bool
    )


    for driver_name, raster_path in (
        NATURAL_DRIVERS.items()
    ):

        (
            risk_data,
            risk_valid,
            risk_transform,
            _
        ) = read_aoi_raster(
            raster_path,
            aoi_geometry,
            aoi.crs
        )


        check_same_grid(
            ecosystem_data,
            reference_transform,
            risk_data,
            risk_transform,
            raster_path.name
        )


        high_risk_mask = (
            disturbed_dryland_mask
            & risk_valid
            & (
                risk_data
                == HIGH_RISK_CLASS
            )
        )


        if np.any(
            high_risk_mask
        ):

            natural_drivers.append(
                driver_name
            )

            natural_driver_mask |= (
                high_risk_mask
            )


    # -------------------------------------------------------------------------
    # 9. Unknown driver
    # -------------------------------------------------------------------------

    unknown_driver_mask = (
        disturbed_dryland_mask
        & ~known_driver_mask
        & ~natural_driver_mask
    )


    if np.any(
        unknown_driver_mask
    ):

        other_drivers.append(
            "Unknown"
        )


    # -------------------------------------------------------------------------
    # 10. Percentages
    # -------------------------------------------------------------------------

    def percent_of_total(area):

        if dryland_total_area_ha == 0:
            return 0

        return (
            area
            / dryland_total_area_ha
            * 100
        )


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "total_area_ha":
            round(
                dryland_total_area_ha,
                2
            ),

        "remaining_forest": {
            "area_ha":
                round(
                    remaining_area_ha,
                    2
                ),

            "percentage":
                round(
                    percent_of_total(
                        remaining_area_ha
                    ),
                    2
                )
        },

        "disturbed": {
            "area_ha":
                round(
                    disturbed_area_ha,
                    2
                ),

            "percentage":
                round(
                    percent_of_total(
                        disturbed_area_ha
                    ),
                    2
                )
        },

        "forest_loss": {
            "area_ha":
                round(
                    forest_loss_area_ha,
                    2
                ),

            "percentage":
                round(
                    percent_of_total(
                        forest_loss_area_ha
                    ),
                    2
                )
        },

        "forest_gain": {
            "area_ha":
                round(
                    forest_gain_area_ha,
                    2
                ),

            "percentage":
                round(
                    percent_of_total(
                        forest_gain_area_ha
                    ),
                    2
                )
        },

        "drivers": {

            "non_natural":
                non_natural_drivers,

            "natural":
                natural_drivers,

            "other":
                other_drivers
        }
    }


# =============================================================================
# Run
# =============================================================================

result = analyze_dryland_forest()


# =============================================================================
# Display
# =============================================================================

print()
print("=" * 75)
print("DRYLAND FOREST")
print("=" * 75)

print(
    f"Total area       : "
    f"{result['total_area_ha']:,.2f} ha"
)

print(
    f"Remaining forest : "
    f"{result['remaining_forest']['area_ha']:,.2f} ha "
    f"({result['remaining_forest']['percentage']:.2f}%)"
)

print(
    f"Disturbed        : "
    f"{result['disturbed']['area_ha']:,.2f} ha "
    f"({result['disturbed']['percentage']:.2f}%)"
)

print(
    f"Forest loss      : "
    f"{result['forest_loss']['area_ha']:,.2f} ha "
    f"({result['forest_loss']['percentage']:.2f}%)"
)

print(
    f"Forest gain      : "
    f"{result['forest_gain']['area_ha']:,.2f} ha "
    f"({result['forest_gain']['percentage']:.2f}%)"
)


print()
print("Dryland forest disturbance drivers")
print("-" * 75)


print()
print("Non-natural drivers:")

if result["drivers"]["non_natural"]:

    for driver in (
        result["drivers"]["non_natural"]
    ):

        print(
            f"  - {driver}"
        )

else:

    print("  None detected")


print()
print("Natural drivers:")

if result["drivers"]["natural"]:

    for driver in (
        result["drivers"]["natural"]
    ):

        print(
            f"  - {driver}"
        )

else:

    print("  None detected")


print()
print("Other drivers:")

if result["drivers"]["other"]:

    for driver in (
        result["drivers"]["other"]
    ):

        print(
            f"  - {driver}"
        )

else:

    print("  None detected")